<a href="https://colab.research.google.com/github/bangaru01/C_programing/blob/main/Cycloetherification_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# R3 CONTROL ANALYSIS
# FRESH CONSOLIDATED VERSION
#
# ============================================================
# INCLUDES
# ============================================================
#
# STERIC
#   1. Sterimol B1
#   2. Sterimol B5
#   3. Sterimol L
#   4. R3-only %Vbur
#   5. R3-only Distal Volume (A^3)
#   6. Geometric cone angle
#   7. Geometric half-cone angle
#   8. C18-C44 distance
#
# SASA
#   9. Total SASA
#  10. Enclosed volume
#  11. SASA at C18
#  12. SASA at C44
#  13. SASA at O17
#
# GEOMETRY
#  14. O17-C22 forming bond
#  15. C22 angle sum
#  16. C22 pyramidalization
#  17. Bürgi-Dunitz O17-C22-C21
#  18. Bürgi-Dunitz O17-C22-C23
#  19. Average Bürgi-Dunitz angle
#  20. Cremer-Pople Q
#  21. Cremer-Pople Theta
#  22. Cremer-Pople Phi
#  23. Ring classification
#  24. O15-H16 distance
#  25. H16-O17 distance
#  26. O15-H16-O17 angle
#  27. C18-R torsion
#
# ELECTRONIC / GFN2-xTB
#  28. Energy
#  29. HOMO
#  30. LUMO
#  31. HOMO-LUMO gap
#  32. Dipole
#  33. IP
#  34. EA
#  35. Chemical potential
#  36. Electronegativity
#  37. Hardness
#  38. Softness
#  39. Electrophilicity
#  40. Nucleophilicity
#  41. Electrofugality
#  42. Nucleofugality
#  43. Fermi level
#  44. Molecular polarizability
#  45. Charges
#  46. Bond order
#  47. Covalent coordination numbers
#  48. Atomic dipoles
#  49. Fukui nucleophilicity
#  50. Fukui electrophilicity
#
# STATISTICS
#  51. Pearson correlations
#  52. Spearman correlations
#  53. Full Pearson matrix
#  54. Full Spearman matrix
#  55. Grouped correlations
#
# OUTPUT
#  56. Pearson heatmap
#  57. Spearman heatmap
#  58. Complete Excel workbook
#  59. Descriptor QC
#  60. xTB QC
#  61. %de mapping QC
#
# INPUT:
#   RR_*.xyz
#
# ============================================================


# ============================================================
# 0. BASIC IMPORTS
# ============================================================

import os
import sys
import subprocess
import warnings
import importlib
import math

warnings.filterwarnings("ignore")

print("=" * 110)
print("R3 CONTROL ANALYSIS")
print("=" * 110)
print("Starting...")


# ============================================================
# 1. INSTALL BASIC PYTHON PACKAGES
# ============================================================

print("\n")
print("=" * 110)
print("CHECKING BASIC PYTHON PACKAGES")
print("=" * 110)

basic_packages = [
    "morfeus-ml",
    "pandas",
    "numpy",
    "scipy",
    "openpyxl",
    "matplotlib",
    "seaborn"
]

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--upgrade"
] + basic_packages)

print("✓ Basic packages installed/updated.")


# ============================================================
# 2. IMPORT BASIC PACKAGES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr, spearmanr

print("✓ NumPy imported.")
print("✓ Pandas imported.")
print("✓ SciPy imported.")
print("✓ Matplotlib imported.")
print("✓ Seaborn imported.")


# ============================================================
# 3. GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive")



# ============================================================
# 4. MORFEUS
# ============================================================

print("\n")
print("=" * 110)
print("MORFEUS")
print("=" * 110)

try:

    import morfeus

    from morfeus import (
        read_xyz,
        Sterimol,
        BuriedVolume,
        XTB,
        SASA
    )

    print(
        "Morfeus version:",
        getattr(
            morfeus,
            "__version__",
            "unknown"
        )
    )

    print("✓ Morfeus imported.")

except Exception as e:

    raise RuntimeError(
        "Morfeus import failed:\n"
        + str(e)
    )


# ============================================================
# 5. xTB PYTHON BACKEND
#
# IMPORTANT:
# Do NOT use:
#
#     pip install xtb
#
# We first test whether xtb is already available.
#
# If not, we attempt a conda-forge installation using
# micromamba.
# ============================================================

print("\n")
print("=" * 110)
print("CHECKING PYTHON xTB BACKEND")
print("=" * 110)


def try_import_xtb():

    try:

        import xtb

        print(
            "✓ Python xtb backend already available."
        )

        return True

    except Exception:

        return False


xtb_available = try_import_xtb()


# ------------------------------------------------------------
# Attempt automatic installation if necessary
# ------------------------------------------------------------

if not xtb_available:

    print(
        "Python xtb backend was not found."
    )

    print(
        "Attempting conda-forge installation..."
    )

    try:

        # ----------------------------------------------------
        # Install micromamba
        # ----------------------------------------------------

        micromamba_binary = (
            "/content/micromamba"
        )

        if not os.path.exists(
            micromamba_binary
        ):

            subprocess.run(
                [
                    "wget",
                    "-qO",
                    "/tmp/micromamba.tar.bz2",
                    "https://micro.mamba.pm/api/micromamba/linux-64/latest"
                ],
                check=True
            )

            os.makedirs(
                "/tmp/micromamba_extract",
                exist_ok=True
            )

            subprocess.run(
                [
                    "tar",
                    "-xjf",
                    "/tmp/micromamba.tar.bz2",
                    "-C",
                    "/tmp/micromamba_extract"
                ],
                check=True
            )

            source_binary = (
                "/tmp/micromamba_extract/"
                "bin/micromamba"
            )

            subprocess.run(
                [
                    "cp",
                    source_binary,
                    micromamba_binary
                ],
                check=True
            )

            os.chmod(
                micromamba_binary,
                0o755
            )

        print(
            "✓ micromamba available."
        )


        # ----------------------------------------------------
        # xTB environment
        # ----------------------------------------------------

        XTB_ENV = (
            "/content/xtb-python-env"
        )


        if not os.path.exists(
            XTB_ENV
        ):

            print(
                "Creating xTB environment..."
            )

            subprocess.run(
                [
                    micromamba_binary,
                    "create",
                    "-y",
                    "-p",
                    XTB_ENV,
                    "-c",
                    "conda-forge",
                    "python=3.11",
                    "xtb-python"
                ],
                check=True
            )

            print(
                "✓ xTB environment created."
            )

        else:

            print(
                "✓ Existing xTB environment found."
            )


        # ----------------------------------------------------
        # Add Python site-packages
        # ----------------------------------------------------

        site_packages = os.path.join(
            XTB_ENV,
            "lib",
            "python3.11",
            "site-packages"
        )


        if os.path.isdir(
            site_packages
        ):

            if site_packages not in sys.path:

                sys.path.insert(
                    0,
                    site_packages
                )

            os.environ["PYTHONPATH"] = (
                site_packages
                +
                os.pathsep
                +
                os.environ.get(
                    "PYTHONPATH",
                    ""
                )
            )


        # ----------------------------------------------------
        # Add native libraries
        # ----------------------------------------------------

        lib_directory = os.path.join(
            XTB_ENV,
            "lib"
        )


        if os.path.isdir(
            lib_directory
        ):

            current_ld = os.environ.get(
                "LD_LIBRARY_PATH",
                ""
            )

            os.environ["LD_LIBRARY_PATH"] = (
                lib_directory
                +
                os.pathsep
                +
                current_ld
            )


        # ----------------------------------------------------
        # Try import again
        # ----------------------------------------------------

        importlib.invalidate_caches()

        try:

            import xtb

            print(
                "✓ Python xtb backend imported."
            )

            xtb_available = True

        except Exception as e:

            print(
                "\nWARNING:"
            )

            print(
                "xTB Python backend could not be imported."
            )

            print(
                str(e)
            )

            xtb_available = False


    except Exception as e:

        print(
            "\nWARNING: Automatic xTB installation failed."
        )

        print(
            type(e).__name__,
            str(e)
        )

        xtb_available = False


# ------------------------------------------------------------
# Final xTB status
# ------------------------------------------------------------

if xtb_available:

    print(
        "\n✓ xTB backend is available."
    )

else:

    print(
        "\n⚠ xTB backend is NOT available."
    )

    print(
        "The structural/steric/SASA/geometric analysis will"
    )

    print(
        "continue, but xTB descriptors will be NaN."
    )


# ============================================================
# 6. PATHS
# ============================================================

xyz_folder = (
    "/content/drive/MyDrive/Cycloetherification_xyz_files"
)

output_folder = os.path.join(
    xyz_folder,
    "R3_Control_Analysis"
)

os.makedirs(
    output_folder,
    exist_ok=True
)

print("\n")
print("=" * 110)
print("PATHS")
print("=" * 110)

print(
    "XYZ folder:",
    xyz_folder
)

print(
    "Output folder:",
    output_folder
)


if not os.path.isdir(
    xyz_folder
):

    raise FileNotFoundError(
        "XYZ folder does not exist:\n"
        +
        xyz_folder
    )


# ============================================================
# 7. FIXED ATOM NUMBERING
# ============================================================

O15 = 15
H16 = 16
O17 = 17

C18 = 18
C19 = 19
C20 = 20
C21 = 21
C22 = 22
C23 = 23

C44 = 44
C45 = 45


# Six-membered ring
RING = [
    17,
    18,
    19,
    20,
    21,
    22
]


# ============================================================
# 8. EXPERIMENTAL %de
# ============================================================

de_data = {

    "CH2-Ph": 56,
    "CH2-dioxane": 34,
    "CH2CN": 60,
    "CH2CO2Me": 42,
    "CMe2CO2Me": 92,
    "Cy": 72,
    "Et": 56,
    "Me-Propane": 60,
    "Me": 56,
    "Ph": 34,
    "alkene": 34,
    "alkyne": 0,
    "allyl": 34,
    "dimethyl-allyl": 90,
    "iPr": 72,
    "nBu": 56,
    "tBu": 98
}


# ============================================================
# 9. SASA SETTINGS
# ============================================================

DCM_PROBE_RADIUS = 2.40


# ============================================================
# 10. HELPER: VALID ATOM
# ============================================================

def validate_atom_number(
    atom_number,
    n_atoms
):

    if (
        atom_number < 1
        or
        atom_number > n_atoms
    ):

        raise IndexError(
            f"Atom {atom_number} is outside "
            f"a structure containing {n_atoms} atoms."
        )


# ============================================================
# 11. GENERAL GEOMETRY FUNCTIONS
# ============================================================

def pos(
    atoms,
    atom
):

    validate_atom_number(
        atom,
        len(atoms)
    )

    return np.asarray(
        atoms[atom - 1][1],
        dtype=float
    )


def distance(
    atoms,
    i,
    j
):

    return float(
        np.linalg.norm(
            pos(atoms, i)
            -
            pos(atoms, j)
        )
    )


def angle(
    atoms,
    i,
    j,
    k
):

    v1 = (
        pos(atoms, i)
        -
        pos(atoms, j)
    )

    v2 = (
        pos(atoms, k)
        -
        pos(atoms, j)
    )

    denominator = (
        np.linalg.norm(v1)
        *
        np.linalg.norm(v2)
    )

    if denominator < 1e-12:

        return np.nan

    cosang = (
        np.dot(v1, v2)
        /
        denominator
    )

    cosang = np.clip(
        cosang,
        -1.0,
        1.0
    )

    return float(
        np.degrees(
            np.arccos(cosang)
        )
    )


def dihedral(
    atoms,
    i,
    j,
    k,
    l
):

    p0 = pos(atoms, i)
    p1 = pos(atoms, j)
    p2 = pos(atoms, k)
    p3 = pos(atoms, l)

    b0 = p0 - p1
    b1 = p2 - p1
    b2 = p3 - p2

    b1_norm = np.linalg.norm(
        b1
    )

    if b1_norm < 1e-12:

        return np.nan

    b1 = (
        b1
        /
        b1_norm
    )

    v = (
        b0
        -
        np.dot(b0, b1) * b1
    )

    w = (
        b2
        -
        np.dot(b2, b1) * b1
    )

    x = np.dot(
        v,
        w
    )

    y = np.dot(
        np.cross(
            b1,
            v
        ),
        w
    )

    return float(
        np.degrees(
            np.arctan2(
                y,
                x
            )
        )
    )


# ============================================================
# 12. STERIMOL
# ============================================================

def calculate_sterimol(
    elements,
    coordinates
):

    n_atoms = len(
        elements
    )

    if n_atoms < C45:

        raise ValueError(
            f"{n_atoms} atoms found; "
            f"at least {C45} atoms are required."
        )


    # R3 starts at C44
    R3_atoms = list(
        range(
            C44,
            n_atoms + 1
        )
    )


    # Keep only C18 + R3 atoms
    excluded_atoms = [

        i

        for i in range(
            1,
            n_atoms + 1
        )

        if (
            i != C18
            and
            i not in R3_atoms
        )
    ]


    sterimol = Sterimol(

        elements,
        coordinates,

        dummy_index=C18,
        attached_index=C44,

        excluded_atoms=excluded_atoms,

        radii_type="crc",

        n_rot_vectors=3600
    )


    return {

        "Sterimol_B1":
            float(
                sterimol.B_1_value
            ),

        "Sterimol_B5":
            float(
                sterimol.B_5_value
            ),

        "Sterimol_L":
            float(
                sterimol.L_value
            )
    }


# ============================================================
# 13. R3 %VBUR + DISTAL VOLUME
#
# IMPORTANT:
#
# Center:
#     C18
#
# R3 group:
#     C44 onward
#
# Everything else is excluded.
#
# Distal volume is attempted using the Morfeus
# compute_distal_volume() method.
#
# If the installed Morfeus version does not provide
# distal volume, the analysis does NOT crash.
# It records NaN and reports the reason.
# ============================================================

def calculate_vbur_and_distal_volume(
    elements,
    coordinates
):

    n_atoms = len(
        elements
    )

    if n_atoms < C44:

        raise ValueError(
            f"{n_atoms} atoms found; "
            f"C44 is required."
        )


    R3_atoms = list(
        range(
            C44,
            n_atoms + 1
        )
    )


    excluded_atoms = [

        i

        for i in range(
            1,
            n_atoms + 1
        )

        if (
            i != C18
            and
            i not in R3_atoms
        )
    ]


    buried = BuriedVolume(

        elements,
        coordinates,

        metal_index=C18,

        excluded_atoms=excluded_atoms,

        radius=3.5,

        include_hs=False,

        radii_type="bondi",

        radii_scale=1.17
    )


    # --------------------------------------------------------
    # %Vbur
    # --------------------------------------------------------

    fraction = getattr(
        buried,
        "fraction_buried_volume",
        np.nan
    )


    try:

        vbur_percent = (
            float(fraction)
            *
            100.0
        )

    except Exception:

        vbur_percent = np.nan


    # --------------------------------------------------------
    # DISTAL VOLUME
    # --------------------------------------------------------

    distal_volume = np.nan

    distal_status = (
        "NOT_AVAILABLE"
    )


    # Try documented/computational method first
    compute_method = getattr(
        buried,
        "compute_distal_volume",
        None
    )


    if callable(
        compute_method
    ):

        try:

            compute_method()

            value = getattr(
                buried,
                "distal_volume",
                np.nan
            )

            if value is not None:

                distal_volume = float(
                    value
                )

                distal_status = (
                    "SUCCESS"
                )

        except Exception as e:

            distal_status = (
                "FAILED: "
                +
                type(e).__name__
                +
                ": "
                +
                str(e)[:100]
            )

    else:

        # ----------------------------------------------------
        # Fallback: check whether distal_volume is already
        # exposed as a property.
        # ----------------------------------------------------

        try:

            value = getattr(
                buried,
                "distal_volume",
                np.nan
            )

            if value is not None:

                value = float(
                    value
                )

                if np.isfinite(
                    value
                ):

                    distal_volume = value

                    distal_status = (
                        "SUCCESS"
                    )

        except Exception:

            distal_status = (
                "NOT_SUPPORTED_BY_MORFEUS"
            )


    return {

        "R3_%Vbur":
            vbur_percent,

        "R3_Distal_Volume_A3":
            distal_volume,

        "Distal_Volume_Status":
            distal_status
    }


# ============================================================
# 14. ROBUST ATOM VALUE EXTRACTION
# ============================================================

def get_atom_value(
    data,
    atom_number
):

    if data is None:

        return np.nan


    # --------------------------------------------------------
    # Dictionary-like object
    # --------------------------------------------------------

    if hasattr(
        data,
        "get"
    ):

        try:

            value = data.get(
                atom_number,
                np.nan
            )

            if value is None:

                return np.nan


            if np.isscalar(
                value
            ):

                return float(
                    value
                )


            arr = np.asarray(
                value,
                dtype=float
            )


            if arr.ndim == 0:

                return float(
                    arr
                )


            return float(
                np.linalg.norm(
                    arr
                )
            )

        except Exception:

            pass


    # --------------------------------------------------------
    # Array/list
    # --------------------------------------------------------

    try:

        arr = np.asarray(
            data
        )


        if arr.ndim == 1:

            if len(arr) >= atom_number:

                return float(
                    arr[atom_number - 1]
                )


        elif arr.ndim >= 2:

            if len(arr) >= atom_number:

                value = arr[
                    atom_number - 1
                ]


                if np.isscalar(
                    value
                ):

                    return float(
                        value
                    )


                return float(
                    np.linalg.norm(
                        value
                    )
                )

    except Exception:

        pass


    return np.nan


# ============================================================
# 15. SASA
# ============================================================

def calculate_sasa(
    elements,
    coordinates,
    probe_radius=DCM_PROBE_RADIUS
):

    sasa = SASA(

        elements,
        coordinates,

        probe_radius=probe_radius
    )


    atom_areas = getattr(
        sasa,
        "atom_areas",
        None
    )


    return {

        "Total_SASA":
            float(
                sasa.area
            ),

        "Enclosed_Volume":
            float(
                sasa.volume
            ),

        "SASA_C18":
            get_atom_value(
                atom_areas,
                C18
            ),

        "SASA_C44":
            get_atom_value(
                atom_areas,
                C44
            ),

        "SASA_O17":
            get_atom_value(
                atom_areas,
                O17
            )
    }


# ============================================================
# 16. GEOMETRIC CONE ANGLE
# ============================================================

def calculate_geometric_cone_angle(
    elements,
    coordinates,
    heavy_atoms_only=True
):

    n_atoms = len(
        elements
    )


    if n_atoms < C45:

        return {

            "Geometric_Cone_Angle":
                np.nan,

            "Geometric_Half_Cone_Angle":
                np.nan,

            "Cone_Limiting_Atom":
                np.nan
        }


    r18 = np.asarray(
        coordinates[C18 - 1],
        dtype=float
    )

    r44 = np.asarray(
        coordinates[C44 - 1],
        dtype=float
    )


    axis = (
        r44
        -
        r18
    )


    axis_length = np.linalg.norm(
        axis
    )


    if axis_length < 1e-12:

        return {

            "Geometric_Cone_Angle":
                np.nan,

            "Geometric_Half_Cone_Angle":
                np.nan,

            "Cone_Limiting_Atom":
                np.nan
        }


    u = (
        axis
        /
        axis_length
    )


    angles = []


    # R3 atoms from C45 onward
    # C44 is the attached atom defining the axis.

    for atom in range(
        C45,
        n_atoms + 1
    ):

        element = str(
            elements[atom - 1]
        ).strip().upper()


        if (
            heavy_atoms_only
            and
            element == "H"
        ):

            continue


        v = (
            np.asarray(
                coordinates[atom - 1],
                dtype=float
            )
            -
            r18
        )


        v_norm = np.linalg.norm(
            v
        )


        if v_norm < 1e-12:

            continue


        cosang = (
            np.dot(
                v,
                u
            )
            /
            v_norm
        )


        cosang = np.clip(
            cosang,
            -1.0,
            1.0
        )


        theta = float(
            np.degrees(
                np.arccos(
                    cosang
                )
            )
        )


        angles.append(
            (
                theta,
                atom
            )
        )


    if not angles:

        return {

            "Geometric_Cone_Angle":
                np.nan,

            "Geometric_Half_Cone_Angle":
                np.nan,

            "Cone_Limiting_Atom":
                np.nan
        }


    limiting_angle, limiting_atom = max(
        angles,
        key=lambda x: x[0]
    )


    return {

        "Geometric_Cone_Angle":
            float(
                2.0 * limiting_angle
            ),

        "Geometric_Half_Cone_Angle":
            float(
                limiting_angle
            ),

        "Cone_Limiting_Atom":
            int(
                limiting_atom
            )
    }


# ============================================================
# 17. FORMING BOND
# ============================================================

def forming_bond(
    atoms
):

    return distance(
        atoms,
        O17,
        C22
    )


# ============================================================
# 18. C22 PLANARITY / PYRAMIDALIZATION
# ============================================================

def c22_planarity(
    atoms
):

    values = [

        angle(
            atoms,
            O17,
            C22,
            C21
        ),

        angle(
            atoms,
            O17,
            C22,
            C23
        ),

        angle(
            atoms,
            C21,
            C22,
            C23
        )
    ]


    if any(
        np.isnan(v)
        for v in values
    ):

        return {

            "C22_Angle_Sum":
                np.nan,

            "C22_Pyramidalization":
                np.nan
        }


    angle_sum = sum(
        values
    )


    return {

        "C22_Angle_Sum":
            float(
                angle_sum
            ),

        "C22_Pyramidalization":
            float(
                360.0 - angle_sum
            )
    }


# ============================================================
# 19. BURGI-DUNITZ
# ============================================================

def burgi_dunitz(
    atoms
):

    bd1 = angle(
        atoms,
        O17,
        C22,
        C21
    )


    bd2 = angle(
        atoms,
        O17,
        C22,
        C23
    )


    valid = [

        x

        for x in [
            bd1,
            bd2
        ]

        if (
            not np.isnan(x)
        )
    ]


    return {

        "BD_O17_C22_C21":
            bd1,

        "BD_O17_C22_C23":
            bd2,

        "BD_Average":
            (
                float(
                    np.mean(valid)
                )
                if valid
                else np.nan
            )
    }


# ============================================================
# 20. PROTON SHUTTLE
# ============================================================

def proton_shuttle(
    atoms
):

    return {

        "O15-H16":
            distance(
                atoms,
                O15,
                H16
            ),

        "H16-O17":
            distance(
                atoms,
                H16,
                O17
            ),

        "O15-H16-O17_Angle":
            angle(
                atoms,
                O15,
                H16,
                O17
            )
    }


# ============================================================
# 21. C18-R ROTAMER
# ============================================================

def c18_r_rotamer(
    atoms
):

    if len(atoms) < C45:

        return {

            "C18-C44_Distance":
                np.nan,

            "C18-R_Torsion":
                np.nan
        }


    return {

        "C18-C44_Distance":
            distance(
                atoms,
                C18,
                C44
            ),

        "C18-R_Torsion":
            dihedral(
                atoms,
                O17,
                C18,
                C44,
                C45
            )
    }


# ============================================================
# 22. CREMER-POPLE 6-MEMBERED RING
# ============================================================

def cremer_pople_6(
    atoms
):

    try:

        coords = np.array(
            [
                pos(
                    atoms,
                    atom
                )

                for atom in RING
            ],
            dtype=float
        )

    except Exception:

        return {

            "CP_Q":
                np.nan,

            "CP_Theta":
                np.nan,

            "CP_Phi":
                np.nan
        }


    if coords.shape != (
        6,
        3
    ):

        return {

            "CP_Q":
                np.nan,

            "CP_Theta":
                np.nan,

            "CP_Phi":
                np.nan
        }


    N = 6


    center = coords.mean(
        axis=0
    )


    centered = (
        coords
        -
        center
    )


    j = np.arange(
        N,
        dtype=float
    )


    # --------------------------------------------------------
    # Reference vectors
    # --------------------------------------------------------

    Rprime = (

        centered

        *

        np.sin(
            2.0
            *
            np.pi
            *
            j
            /
            N
        )[:, None]

    ).sum(
        axis=0
    )


    Rdouble = (

        centered

        *

        np.cos(
            2.0
            *
            np.pi
            *
            j
            /
            N
        )[:, None]

    ).sum(
        axis=0
    )


    normal = np.cross(
        Rprime,
        Rdouble
    )


    norm = np.linalg.norm(
        normal
    )


    if norm < 1e-12:

        return {

            "CP_Q":
                np.nan,

            "CP_Theta":
                np.nan,

            "CP_Phi":
                np.nan
        }


    normal = (
        normal
        /
        norm
    )


    z = (
        centered
        @
        normal
    )


    # --------------------------------------------------------
    # m = 2
    # --------------------------------------------------------

    c = (

        np.sqrt(
            2.0 / N
        )

        *

        np.sum(

            z

            *

            np.cos(
                4.0
                *
                np.pi
                *
                j
                /
                N
            )
        )
    )


    s = (

        -np.sqrt(
            2.0 / N
        )

        *

        np.sum(

            z

            *

            np.sin(
                4.0
                *
                np.pi
                *
                j
                /
                N
            )
        )
    )


    q2 = np.sqrt(
        c**2
        +
        s**2
    )


    # --------------------------------------------------------
    # m = 3
    # --------------------------------------------------------

    q3 = (

        np.sqrt(
            1.0 / N
        )

        *

        np.sum(

            z

            *

            (
                (-1.0) ** j
            )
        )
    )


    Q = np.sqrt(
        q2**2
        +
        q3**2
    )


    if Q > 1e-12:

        theta = np.degrees(
            np.arccos(
                np.clip(
                    q3 / Q,
                    -1.0,
                    1.0
                )
            )
        )

    else:

        theta = 0.0


    phi = (

        np.degrees(
            np.arctan2(
                s,
                c
            )
        )

        %

        360.0
    )


    return {

        "CP_Q":
            float(Q),

        "CP_Theta":
            float(theta),

        "CP_Phi":
            float(phi)
    }


# ============================================================
# 23. CREMER-POPLE CLASSIFICATION
# ============================================================

def classify_ring(
    theta,
    phi
):

    if (
        theta is None
        or
        phi is None
        or
        not np.isfinite(theta)
        or
        not np.isfinite(phi)
    ):

        return "Unknown"


    if (
        theta < 15.0
        or
        theta > 165.0
    ):

        return "Chair"


    if (
        75.0 <= theta <= 105.0
    ):

        m = phi % 60.0

        if (
            m < 15.0
            or
            m > 45.0
        ):

            return "Boat"

        return "Twist-boat"


    if (
        35.0 <= theta <= 65.0
        or
        115.0 <= theta <= 145.0
    ):

        m = phi % 60.0

        if (
            m < 15.0
            or
            m > 45.0
        ):

            return "Envelope"

        return "Half-chair"


    return "Intermediate"


# ============================================================
# 24. COMPLETE GEOMETRY
# ============================================================

def analyze_geometry(
    atoms
):

    result = {}


    result[
        "O17-C22_Forming_Bond"
    ] = forming_bond(
        atoms
    )


    result.update(
        c22_planarity(
            atoms
        )
    )


    result.update(
        burgi_dunitz(
            atoms
        )
    )


    cp = cremer_pople_6(
        atoms
    )


    result.update(
        cp
    )


    result[
        "Ring_Classification"
    ] = classify_ring(
        cp[
            "CP_Theta"
        ],
        cp[
            "CP_Phi"
        ]
    )


    result.update(
        proton_shuttle(
            atoms
        )
    )


    result.update(
        c18_r_rotamer(
            atoms
        )
    )


    return result


# ============================================================
# 25. SAFE XTB CALL
# ============================================================

def safe_xtb_call(
    label,
    function,
    default=np.nan
):

    try:

        value = function()


        if value is None:

            return default


        if isinstance(
            value,
            np.generic
        ):

            value = value.item()


        if np.isscalar(
            value
        ):

            try:

                return float(
                    value
                )

            except Exception:

                return value


        return value


    except Exception as e:

        print(
            f"      XTB {label} failed: "
            f"{type(e).__name__}: "
            f"{str(e)[:150]}"
        )

        return default


# ============================================================
# 26. XTB DESCRIPTOR COLUMNS
# ============================================================

XTB_DESCRIPTOR_COLUMNS = [

    "XTB_Energy",

    "XTB_HOMO",
    "XTB_LUMO",
    "XTB_Gap",

    "XTB_Dipole",

    "XTB_IP",
    "XTB_EA",

    "XTB_Chemical_Potential",
    "XTB_Electronegativity",

    "XTB_Hardness",
    "XTB_Softness",

    "XTB_Electrophilicity",
    "XTB_Nucleophilicity",

    "XTB_Electrofugality",
    "XTB_Nucleofugality",

    "XTB_Fermi_Level",

    "XTB_Molecular_Polarizability",

    "XTB_Charge_C18",
    "XTB_Charge_C44",
    "XTB_Charge_C22",
    "XTB_Charge_O17",

    "XTB_BondOrder_O17_C22",

    "XTB_CovCN_C18",
    "XTB_CovCN_C44",
    "XTB_CovCN_C22",
    "XTB_CovCN_O17",

    "XTB_AtomDipole_C18",
    "XTB_AtomDipole_C44",
    "XTB_AtomDipole_C22",
    "XTB_AtomDipole_O17",

    "XTB_Fukui_Nucleophilicity_C18",
    "XTB_Fukui_Nucleophilicity_C44",
    "XTB_Fukui_Nucleophilicity_C22",
    "XTB_Fukui_Nucleophilicity_O17",

    "XTB_Fukui_Electrophilicity_C18",
    "XTB_Fukui_Electrophilicity_C44",
    "XTB_Fukui_Electrophilicity_C22",
    "XTB_Fukui_Electrophilicity_O17"
]


# ============================================================
# 27. COMPLETE GFN2-xTB
# ============================================================

def calculate_xtb_descriptors(
    elements,
    coordinates
):

    result = {}

    # Initialize every expected descriptor
    for col in XTB_DESCRIPTOR_COLUMNS:

        result[col] = np.nan


    # --------------------------------------------------------
    # If xTB unavailable
    # --------------------------------------------------------

    if not xtb_available:

        return result


    print(
        "    Running GFN2-xTB..."
    )


    # --------------------------------------------------------
    # Create Morfeus XTB object
    # --------------------------------------------------------

    xtb_calc = XTB(

        elements,

        coordinates,

        method=2
    )


    # ========================================================
    # GLOBAL ELECTRONIC
    # ========================================================

    result[
        "XTB_Energy"
    ] = safe_xtb_call(
        "Energy",
        lambda:
            xtb_calc.get_energy()
    )


    result[
        "XTB_HOMO"
    ] = safe_xtb_call(
        "HOMO",
        lambda:
            xtb_calc.get_homo(
                unit="eV"
            )
    )


    result[
        "XTB_LUMO"
    ] = safe_xtb_call(
        "LUMO",
        lambda:
            xtb_calc.get_lumo(
                unit="eV"
            )
    )


    result[
        "XTB_Gap"
    ] = safe_xtb_call(
        "HOMO-LUMO Gap",
        lambda:
            xtb_calc.get_homo_lumo_gap(
                unit="eV"
            )
    )


    result[
        "XTB_Dipole"
    ] = safe_xtb_call(
        "Dipole",
        lambda:
            xtb_calc.get_dipole_moment(
                unit="debye"
            )
    )


    result[
        "XTB_IP"
    ] = safe_xtb_call(
        "Ionization Potential",
        lambda:
            xtb_calc.get_ip(
                corrected=True
            )
    )


    result[
        "XTB_EA"
    ] = safe_xtb_call(
        "Electron Affinity",
        lambda:
            xtb_calc.get_ea(
                corrected=True
            )
    )


    result[
        "XTB_Chemical_Potential"
    ] = safe_xtb_call(
        "Chemical Potential",
        lambda:
            xtb_calc.get_chemical_potential()
    )


    result[
        "XTB_Electronegativity"
    ] = safe_xtb_call(
        "Electronegativity",
        lambda:
            xtb_calc.get_electronegativity()
    )


    result[
        "XTB_Hardness"
    ] = safe_xtb_call(
        "Hardness",
        lambda:
            xtb_calc.get_hardness()
    )


    result[
        "XTB_Softness"
    ] = safe_xtb_call(
        "Softness",
        lambda:
            xtb_calc.get_softness()
    )


    result[
        "XTB_Electrophilicity"
    ] = safe_xtb_call(
        "Electrophilicity",
        lambda:
            xtb_calc.get_global_descriptor(
                "electrophilicity"
            )
    )


    result[
        "XTB_Nucleophilicity"
    ] = safe_xtb_call(
        "Nucleophilicity",
        lambda:
            xtb_calc.get_global_descriptor(
                "nucleophilicity"
            )
    )


    result[
        "XTB_Electrofugality"
    ] = safe_xtb_call(
        "Electrofugality",
        lambda:
            xtb_calc.get_global_descriptor(
                "electrofugality"
            )
    )


    result[
        "XTB_Nucleofugality"
    ] = safe_xtb_call(
        "Nucleofugality",
        lambda:
            xtb_calc.get_global_descriptor(
                "nucleofugality"
            )
    )


    result[
        "XTB_Fermi_Level"
    ] = safe_xtb_call(
        "Fermi Level",
        lambda:
            xtb_calc.get_fermi_level()
    )


    result[
        "XTB_Molecular_Polarizability"
    ] = safe_xtb_call(
        "Molecular Polarizability",
        lambda:
            xtb_calc.get_molecular_polarizability()
    )


    # ========================================================
    # ATOMIC CHARGES
    # ========================================================

    try:

        charges = (
            xtb_calc.get_charges()
        )


        result[
            "XTB_Charge_C18"
        ] = get_atom_value(
            charges,
            C18
        )


        result[
            "XTB_Charge_C44"
        ] = get_atom_value(
            charges,
            C44
        )


        result[
            "XTB_Charge_C22"
        ] = get_atom_value(
            charges,
            C22
        )


        result[
            "XTB_Charge_O17"
        ] = get_atom_value(
            charges,
            O17
        )


    except Exception as e:

        print(
            "      Atomic charges failed:",
            type(e).__name__,
            str(e)[:150]
        )


    # ========================================================
    # BOND ORDER
    # ========================================================

    try:

        bond_order = (
            xtb_calc.get_bond_order()
        )


        # Try dictionary representation
        bo_value = np.nan


        if hasattr(
            bond_order,
            "get"
        ):

            try:

                bo_value = bond_order.get(
                    (
                        O17,
                        C22
                    ),
                    np.nan
                )

            except Exception:

                bo_value = np.nan


            if (
                not np.isfinite(
                    bo_value
                )
                and
                hasattr(
                    bond_order,
                    "get"
                )
            ):

                try:

                    bo_value = bond_order.get(
                        (
                            C22,
                            O17
                        ),
                        np.nan
                    )

                except Exception:

                    pass


        # Try 2D array
        if not np.isfinite(
            bo_value
        ):

            try:

                bo_array = np.asarray(
                    bond_order,
                    dtype=float
                )


                if (
                    bo_array.ndim == 2
                    and
                    bo_array.shape[0] >= C22
                    and
                    bo_array.shape[1] >= C22
                ):

                    bo_value = bo_array[
                        O17 - 1,
                        C22 - 1
                    ]

            except Exception:

                pass


        result[
            "XTB_BondOrder_O17_C22"
        ] = float(
            bo_value
        ) if np.isfinite(
            bo_value
        ) else np.nan


    except Exception as e:

        print(
            "      Bond order failed:",
            type(e).__name__,
            str(e)[:150]
        )


    # ========================================================
    # COVALENT COORDINATION NUMBERS
    # ========================================================

    try:

        covcn = (
            xtb_calc.get_covcn()
        )


        result[
            "XTB_CovCN_C18"
        ] = get_atom_value(
            covcn,
            C18
        )


        result[
            "XTB_CovCN_C44"
        ] = get_atom_value(
            covcn,
            C44
        )


        result[
            "XTB_CovCN_C22"
        ] = get_atom_value(
            covcn,
            C22
        )


        result[
            "XTB_CovCN_O17"
        ] = get_atom_value(
            covcn,
            O17
        )


    except Exception as e:

        print(
            "      CovCN failed:",
            type(e).__name__,
            str(e)[:150]
        )


    # ========================================================
    # ATOMIC DIPOLES
    # ========================================================

    try:

        atom_dipoles = (
            xtb_calc.get_atom_dipole_moments(
                unit="debye"
            )
        )


        result[
            "XTB_AtomDipole_C18"
        ] = get_atom_value(
            atom_dipoles,
            C18
        )


        result[
            "XTB_AtomDipole_C44"
        ] = get_atom_value(
            atom_dipoles,
            C44
        )


        result[
            "XTB_AtomDipole_C22"
        ] = get_atom_value(
            atom_dipoles,
            C22
        )


        result[
            "XTB_AtomDipole_O17"
        ] = get_atom_value(
            atom_dipoles,
            O17
        )


    except Exception as e:

        print(
            "      Atomic dipoles failed:",
            type(e).__name__,
            str(e)[:150]
        )


    # ========================================================
    # FUKUI NUCLEOPHILICITY
    # ========================================================

    try:

        fukui_minus = (
            xtb_calc.get_fukui(
                "nucleophilicity",
                corrected=True
            )
        )


        result[
            "XTB_Fukui_Nucleophilicity_C18"
        ] = get_atom_value(
            fukui_minus,
            C18
        )


        result[
            "XTB_Fukui_Nucleophilicity_C44"
        ] = get_atom_value(
            fukui_minus,
            C44
        )


        result[
            "XTB_Fukui_Nucleophilicity_C22"
        ] = get_atom_value(
            fukui_minus,
            C22
        )


        result[
            "XTB_Fukui_Nucleophilicity_O17"
        ] = get_atom_value(
            fukui_minus,
            O17
        )


    except Exception as e:

        print(
            "      Nucleophilic Fukui failed:",
            type(e).__name__,
            str(e)[:150]
        )


    # ========================================================
    # FUKUI ELECTROPHILICITY
    # ========================================================

    try:

        fukui_plus = (
            xtb_calc.get_fukui(
                "electrophilicity",
                corrected=True
            )
        )


        result[
            "XTB_Fukui_Electrophilicity_C18"
        ] = get_atom_value(
            fukui_plus,
            C18
        )


        result[
            "XTB_Fukui_Electrophilicity_C44"
        ] = get_atom_value(
            fukui_plus,
            C44
        )


        result[
            "XTB_Fukui_Electrophilicity_C22"
        ] = get_atom_value(
            fukui_plus,
            C22
        )


        result[
            "XTB_Fukui_Electrophilicity_O17"
        ] = get_atom_value(
            fukui_plus,
            O17
        )


    except Exception as e:

        print(
            "      Electrophilic Fukui failed:",
            type(e).__name__,
            str(e)[:150]
        )


    return result


# ============================================================
# 28. ANALYZE ONE XYZ FILE
# ============================================================

def analyze_xyz_file(
    filepath
):

    # --------------------------------------------------------
    # Read XYZ
    # --------------------------------------------------------

    elements, coordinates = read_xyz(
        filepath
    )


    elements = list(
        elements
    )


    coordinates = np.asarray(
        coordinates,
        dtype=float
    )


    if len(elements) != len(
        coordinates
    ):

        raise ValueError(
            "Number of elements and coordinates differ."
        )


    if coordinates.ndim != 2:

        raise ValueError(
            "Coordinates must be a 2D array."
        )


    if coordinates.shape[1] != 3:

        raise ValueError(
            "Coordinates must have exactly 3 columns."
        )


    if len(elements) < C45:

        raise ValueError(
            f"Only {len(elements)} atoms found. "
            f"At least {C45} atoms are required."
        )


    atoms = [

        (
            elements[i],
            coordinates[i]
        )

        for i in range(
            len(elements)
        )
    ]


    filename = os.path.basename(
        filepath
    )


    # --------------------------------------------------------
    # R3 name
    # --------------------------------------------------------

    r3_name = filename


    if r3_name.startswith(
        "RR_"
    ):

        r3_name = r3_name[3:]


    if r3_name.lower().endswith(
        ".xyz"
    ):

        r3_name = r3_name[:-4]


    # --------------------------------------------------------
    # Initial result
    # --------------------------------------------------------

    result = {

        "R3":
            r3_name,

        "Filename":
            filename,

        "%de":
            de_data.get(
                r3_name,
                np.nan
            )
    }


    # ========================================================
    # STERIMOL
    # ========================================================

    try:

        result.update(
            calculate_sterimol(
                elements,
                coordinates
            )
        )

    except Exception as e:

        print(
            "    Sterimol failed:",
            type(e).__name__,
            str(e)[:150]
        )

        result.update({

            "Sterimol_B1":
                np.nan,

            "Sterimol_B5":
                np.nan,

            "Sterimol_L":
                np.nan
        })


    # ========================================================
    # %VBUR + DISTAL VOLUME
    # ========================================================

    try:

        vbur_results = (
            calculate_vbur_and_distal_volume(
                elements,
                coordinates
            )
        )


        result.update(
            vbur_results
        )


    except Exception as e:

        print(
            "    Vbur / Distal Volume failed:",
            type(e).__name__,
            str(e)[:150]
        )


        result.update({

            "R3_%Vbur":
                np.nan,

            "R3_Distal_Volume_A3":
                np.nan,

            "Distal_Volume_Status":
                (
                    "FAILED: "
                    +
                    type(e).__name__
                )
        })


    # ========================================================
    # SASA
    # ========================================================

    try:

        result.update(
            calculate_sasa(
                elements,
                coordinates,
                DCM_PROBE_RADIUS
            )
        )


    except Exception as e:

        print(
            "    SASA failed:",
            type(e).__name__,
            str(e)[:150]
        )


        result.update({

            "Total_SASA":
                np.nan,

            "Enclosed_Volume":
                np.nan,

            "SASA_C18":
                np.nan,

            "SASA_C44":
                np.nan,

            "SASA_O17":
                np.nan
        })


    # ========================================================
    # GEOMETRIC CONE
    # ========================================================

    try:

        result.update(
            calculate_geometric_cone_angle(
                elements,
                coordinates,
                heavy_atoms_only=True
            )
        )


    except Exception as e:

        print(
            "    Cone angle failed:",
            type(e).__name__,
            str(e)[:150]
        )


        result.update({

            "Geometric_Cone_Angle":
                np.nan,

            "Geometric_Half_Cone_Angle":
                np.nan,

            "Cone_Limiting_Atom":
                np.nan
        })


    # ========================================================
    # GEOMETRY
    # ========================================================

    try:

        result.update(
            analyze_geometry(
                atoms
            )
        )


    except Exception as e:

        print(
            "    Geometry failed:",
            type(e).__name__,
            str(e)[:150]
        )


        result.update({

            "O17-C22_Forming_Bond":
                np.nan,

            "C22_Angle_Sum":
                np.nan,

            "C22_Pyramidalization":
                np.nan,

            "BD_O17_C22_C21":
                np.nan,

            "BD_O17_C22_C23":
                np.nan,

            "BD_Average":
                np.nan,

            "CP_Q":
                np.nan,

            "CP_Theta":
                np.nan,

            "CP_Phi":
                np.nan,

            "Ring_Classification":
                "Unknown",

            "O15-H16":
                np.nan,

            "H16-O17":
                np.nan,

            "O15-H16-O17_Angle":
                np.nan,

            "C18-C44_Distance":
                np.nan,

            "C18-R_Torsion":
                np.nan
        })


    # ========================================================
    # XTB
    # ========================================================

    try:

        xtb_result = (
            calculate_xtb_descriptors(
                elements,
                coordinates
            )
        )


        result.update(
            xtb_result
        )


        if xtb_available:

            result[
                "XTB_Status"
            ] = "SUCCESS"

        else:

            result[
                "XTB_Status"
            ] = "UNAVAILABLE"


    except Exception as e:

        print(
            "    XTB completely failed:",
            type(e).__name__,
            str(e)[:200]
        )


        for col in XTB_DESCRIPTOR_COLUMNS:

            result[col] = np.nan


        result[
            "XTB_Status"
        ] = (
            "FAILED: "
            +
            type(e).__name__
        )


    # ========================================================
    # FINAL STATUS
    # ========================================================

    result[
        "Analysis_Status"
    ] = "COMPLETED"


    return result


# ============================================================
# 29. FIND RR FILES
# ============================================================

print("\n")
print("=" * 110)
print("RR XYZ FILE DISCOVERY")
print("=" * 110)


rr_files = sorted([

    f

    for f in os.listdir(
        xyz_folder
    )

    if (
        f.startswith("RR_")
        and
        f.lower().endswith(".xyz")
    )

])


print(
    "Number of RR XYZ files:",
    len(rr_files)
)


for f in rr_files:

    print(
        "   ",
        f
    )


if not rr_files:

    raise FileNotFoundError(
        "No RR_*.xyz files were found in:\n"
        +
        xyz_folder
    )


# ============================================================
# 30. %de MAPPING QC
# ============================================================

file_names_without_extension = []


for filename in rr_files:

    name = filename


    if name.startswith(
        "RR_"
    ):

        name = name[3:]


    if name.lower().endswith(
        ".xyz"
    ):

        name = name[:-4]


    file_names_without_extension.append(
        name
    )


de_mapping_qc = []


for name in file_names_without_extension:

    de_mapping_qc.append({

        "R3":
            name,

        "Experimental_%de":
            de_data.get(
                name,
                np.nan
            ),

        "Mapping_Status":
            (
                "FOUND"
                if name in de_data
                else "MISSING"
            )
    })


de_mapping_df = pd.DataFrame(
    de_mapping_qc
)


print("\n")
print("=" * 110)
print("%de MAPPING QC")
print("=" * 110)


display(
    de_mapping_df
)


missing_de_names = (
    de_mapping_df.loc[
        de_mapping_df[
            "Mapping_Status"
        ] == "MISSING",
        "R3"
    ]
    .tolist()
)


if missing_de_names:

    print(
        "\nWARNING:"
    )

    print(
        "The following structures have no experimental %de:"
    )


    for name in missing_de_names:

        print(
            "   ",
            name
        )


    print(
        "\nThese values will remain NaN."
    )


# ============================================================
# 31. RUN COMPLETE ANALYSIS
# ============================================================

all_results = []


print("\n")
print("=" * 110)
print("STARTING COMPLETE R3 ANALYSIS")
print("=" * 110)


for counter, filename in enumerate(
    rr_files,
    start=1
):

    filepath = os.path.join(
        xyz_folder,
        filename
    )


    print("\n")
    print("-" * 110)

    print(
        f"[{counter}/{len(rr_files)}] "
        f"Processing: {filename}"
    )

    print("-" * 110)


    try:

        result = analyze_xyz_file(
            filepath
        )


        all_results.append(
            result
        )


        print(
            "    ✓ Completed"
        )


    except Exception as e:

        print(
            "    ✗ FAILED:",
            type(e).__name__,
            str(e)
        )


# ============================================================
# 32. CREATE RESULTS DATAFRAME
# ============================================================

if not all_results:

    raise RuntimeError(
        "No XYZ files were successfully analyzed."
    )


results_df = pd.DataFrame(
    all_results
)


# ============================================================
# 33. REMOVE DUPLICATE COLUMNS
# ============================================================

duplicate_columns = (
    results_df.columns[
        results_df.columns.duplicated()
    ]
    .tolist()
)


if duplicate_columns:

    print("\n")
    print(
        "WARNING: Duplicate columns found:"
    )

    print(
        duplicate_columns
    )


    results_df = results_df.loc[
        :,
        ~results_df.columns.duplicated(
            keep="first"
        )
    ]


    print(
        "✓ Duplicate columns removed."
    )


# ============================================================
# 34. SORT RESULTS
# ============================================================

results_df = (
    results_df
    .sort_values(
        "R3"
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 35. XTB QC
# ============================================================

xtb_columns = [

    col

    for col in results_df.columns

    if (
        col.startswith("XTB_")
        and
        col != "XTB_Status"
    )
]


print("\n")
print("=" * 110)
print("XTB QUALITY CONTROL")
print("=" * 110)


print(
    "XTB descriptor columns:",
    len(xtb_columns)
)


print(
    "RR structures:",
    len(results_df)
)


if "XTB_Status" in results_df.columns:

    print(
        "\nXTB status:"
    )


    print(
        results_df[
            [
                "R3",
                "XTB_Status"
            ]
        ]
        .to_string(
            index=False
        )
    )


xtb_missing_df = pd.DataFrame({

    "Descriptor":
        xtb_columns,

    "Missing_Count":
        [
            results_df[
                col
            ]
            .isna()
            .sum()

            for col in xtb_columns
        ],

    "Valid_Count":
        [
            results_df[
                col
            ]
            .notna()
            .sum()

            for col in xtb_columns
        ]
})


print(
    "\nXTB missing values:"
)


print(
    xtb_missing_df.to_string(
        index=False
    )
)


# ============================================================
# 36. COMPLETE DATASET DISPLAY
# ============================================================

print("\n")
print("=" * 120)
print("COMPLETE R3 ANALYSIS")
print("=" * 120)


print(
    "Rows:",
    len(results_df)
)


print(
    "Columns:",
    len(results_df.columns)
)


display(
    results_df
)


# ============================================================
# 37. DESCRIPTOR GROUPS
#
# IMPORTANT:
#
# C18-C44_Distance is ONLY in Steric.
#
# R3_Distal_Volume_A3 is ALSO in Steric.
#
# Cone_Limiting_Atom, Distal_Volume_Status and
# Ring_Classification are QC/categorical fields and are
# NOT entered into numeric correlations.
# ============================================================

steric_descriptors = [

    "Sterimol_B1",
    "Sterimol_B5",
    "Sterimol_L",

    "R3_%Vbur",
    "R3_Distal_Volume_A3",

    "Geometric_Cone_Angle",
    "Geometric_Half_Cone_Angle",

    "C18-C44_Distance"
]


sasa_descriptors = [

    "Total_SASA",
    "Enclosed_Volume",

    "SASA_C18",
    "SASA_C44",
    "SASA_O17"
]


electronic_descriptors = [

    "XTB_Energy",

    "XTB_HOMO",
    "XTB_LUMO",
    "XTB_Gap",

    "XTB_Dipole",

    "XTB_IP",
    "XTB_EA",

    "XTB_Chemical_Potential",
    "XTB_Electronegativity",

    "XTB_Hardness",
    "XTB_Softness",

    "XTB_Electrophilicity",
    "XTB_Nucleophilicity",

    "XTB_Electrofugality",
    "XTB_Nucleofugality",

    "XTB_Fermi_Level",

    "XTB_Molecular_Polarizability",

    "XTB_Charge_C18",
    "XTB_Charge_C44",
    "XTB_Charge_C22",
    "XTB_Charge_O17",

    "XTB_BondOrder_O17_C22",

    "XTB_CovCN_C18",
    "XTB_CovCN_C44",
    "XTB_CovCN_C22",
    "XTB_CovCN_O17",

    "XTB_AtomDipole_C18",
    "XTB_AtomDipole_C44",
    "XTB_AtomDipole_C22",
    "XTB_AtomDipole_O17",

    "XTB_Fukui_Nucleophilicity_C18",
    "XTB_Fukui_Nucleophilicity_C44",
    "XTB_Fukui_Nucleophilicity_C22",
    "XTB_Fukui_Nucleophilicity_O17",

    "XTB_Fukui_Electrophilicity_C18",
    "XTB_Fukui_Electrophilicity_C44",
    "XTB_Fukui_Electrophilicity_C22",
    "XTB_Fukui_Electrophilicity_O17"
]


geometric_descriptors = [

    "O17-C22_Forming_Bond",

    "C22_Angle_Sum",
    "C22_Pyramidalization",

    "BD_O17_C22_C21",
    "BD_O17_C22_C23",
    "BD_Average",

    "CP_Q",
    "CP_Theta",
    "CP_Phi",

    "O15-H16",
    "H16-O17",
    "O15-H16-O17_Angle",

    "C18-R_Torsion"
]


all_descriptor_groups = {

    "Steric":
        steric_descriptors,

    "SASA":
        sasa_descriptors,

    "Electronic":
        electronic_descriptors,

    "Geometric distortion":
        geometric_descriptors
}


# ============================================================
# 38. CREATE UNIQUE DESCRIPTOR LIST
# ============================================================

all_descriptors = []

descriptor_group_membership = []


for group_name, descriptors in (
    all_descriptor_groups.items()
):

    for descriptor in descriptors:

        present = (
            descriptor
            in
            results_df.columns
        )


        if present:

            if descriptor not in all_descriptors:

                all_descriptors.append(
                    descriptor
                )


            descriptor_group_membership.append({

                "Group":
                    group_name,

                "Descriptor":
                    descriptor,

                "Present_in_Data":
                    True
            })


descriptor_summary_df = pd.DataFrame(
    descriptor_group_membership
)


# ============================================================
# 39. DESCRIPTOR QC
# ============================================================

descriptor_qc_rows = []


for group_name, descriptors in (
    all_descriptor_groups.items()
):

    for descriptor in descriptors:

        present = (
            descriptor
            in
            results_df.columns
        )


        if present:

            numeric_values = pd.to_numeric(
                results_df[
                    descriptor
                ],
                errors="coerce"
            )


            valid_n = (
                numeric_values
                .notna()
                .sum()
            )

        else:

            valid_n = 0


        descriptor_qc_rows.append({

            "Group":
                group_name,

            "Descriptor":
                descriptor,

            "Present":
                present,

            "Valid_N":
                int(valid_n),

            "Missing_N":
                int(
                    len(results_df)
                    -
                    valid_n
                )
        })


descriptor_qc_df = pd.DataFrame(
    descriptor_qc_rows
)


print("\n")
print("=" * 110)
print("DESCRIPTOR QC")
print("=" * 110)


print(
    "Unique descriptors:",
    len(all_descriptors)
)


for group_name, descriptors in (
    all_descriptor_groups.items()
):

    valid = [

        d

        for d in descriptors

        if d in results_df.columns
    ]


    print(
        f"\n{group_name}: "
        f"{len(valid)} descriptors"
    )


    for descriptor in valid:

        print(
            "   ",
            descriptor
        )


# ============================================================
# 40. PREPARE CORRELATION DATA
#
# CRITICAL DUPLICATE-COLUMN PROTECTION
# ============================================================

corr_columns = list(
    dict.fromkeys(
        [
            "%de"
        ]
        +
        all_descriptors
    )
)


corr_columns = [

    col

    for col in corr_columns

    if col in results_df.columns
]


corr_data = results_df.loc[
    :,
    corr_columns
].copy()


# Remove duplicate labels
corr_data = corr_data.loc[
    :,
    ~corr_data.columns.duplicated(
        keep="first"
    )
].copy()


# Convert all columns simultaneously
corr_data = corr_data.apply(
    pd.to_numeric,
    errors="coerce"
)


# Remove completely empty columns
empty_corr_columns = [

    col

    for col in corr_data.columns

    if (
        corr_data[
            col
        ]
        .notna()
        .sum()
        ==
        0
    )
]


if empty_corr_columns:

    print("\n")
    print(
        "Descriptors with NO numerical values:"
    )


    for col in empty_corr_columns:

        print(
            "   ",
            col
        )


    corr_data = corr_data.drop(
        columns=empty_corr_columns
    )


print("\n")
print("=" * 110)
print("CORRELATION DATA QC")
print("=" * 110)


print(
    "Rows:",
    len(corr_data)
)


print(
    "Columns:",
    len(corr_data.columns)
)


print(
    "Descriptors entering correlation:",
    max(
        len(corr_data.columns) - 1,
        0
    )
)


# ============================================================
# 41. PEARSON / SPEARMAN WITH %de
# ============================================================

pearson_rows = []
spearman_rows = []


for descriptor in all_descriptors:

    if descriptor not in corr_data.columns:

        continue


    pair = corr_data[
        [
            descriptor,
            "%de"
        ]
    ].dropna()


    n = len(
        pair
    )


    if n < 3:

        pearson_rows.append({

            "Descriptor":
                descriptor,

            "N":
                n,

            "Pearson_r":
                np.nan,

            "Pearson_p":
                np.nan
        })


        spearman_rows.append({

            "Descriptor":
                descriptor,

            "N":
                n,

            "Spearman_rho":
                np.nan,

            "Spearman_p":
                np.nan
        })


        continue


    x = pair[
        descriptor
    ].to_numpy(
        dtype=float
    )


    y = pair[
        "%de"
    ].to_numpy(
        dtype=float
    )


    # --------------------------------------------------------
    # Constant-value protection
    # --------------------------------------------------------

    if (
        np.unique(x).size < 2
        or
        np.unique(y).size < 2
    ):

        pr = np.nan
        pp = np.nan
        sr = np.nan
        sp = np.nan

    else:

        try:

            pr, pp = pearsonr(
                x,
                y
            )

        except Exception:

            pr = np.nan
            pp = np.nan


        try:

            sr, sp = spearmanr(
                x,
                y
            )

        except Exception:

            sr = np.nan
            sp = np.nan


    pearson_rows.append({

        "Descriptor":
            descriptor,

        "N":
            n,

        "Pearson_r":
            pr,

        "Pearson_p":
            pp
    })


    spearman_rows.append({

        "Descriptor":
            descriptor,

        "N":
            n,

        "Spearman_rho":
            sr,

        "Spearman_p":
            sp
    })


pearson_df = pd.DataFrame(
    pearson_rows
)


spearman_df = pd.DataFrame(
    spearman_rows
)


# ============================================================
# 42. COMBINED CORRELATION TABLE
# ============================================================

correlation_df = pd.merge(

    pearson_df,

    spearman_df,

    on=[
        "Descriptor",
        "N"
    ],

    how="outer"
)


if not correlation_df.empty:

    correlation_df[
        "Abs_Pearson_r"
    ] = correlation_df[
        "Pearson_r"
    ].abs()


    correlation_df[
        "Abs_Spearman_rho"
    ] = correlation_df[
        "Spearman_rho"
    ].abs()


    correlation_df = (
        correlation_df
        .sort_values(
            "Abs_Spearman_rho",
            ascending=False,
            na_position="last"
        )
        .reset_index(
            drop=True
        )
    )


print("\n")
print("=" * 110)
print("CORRELATIONS WITH EXPERIMENTAL %de")
print("=" * 110)


display(
    correlation_df
)


# ============================================================
# 43. FULL PEARSON MATRIX
# ============================================================

pearson_matrix = corr_data.corr(
    method="pearson"
)


# ============================================================
# 44. FULL SPEARMAN MATRIX
# ============================================================

spearman_matrix = corr_data.corr(
    method="spearman"
)


print("\n")
print("=" * 110)
print("FULL CORRELATION MATRICES")
print("=" * 110)


print(
    "Pearson matrix:",
    pearson_matrix.shape
)


print(
    "Spearman matrix:",
    spearman_matrix.shape
)


# ============================================================
# 45. HEATMAP FUNCTION
#
# For a large descriptor set, annotation is disabled.
# Exact numbers remain available in Excel.
# ============================================================

def save_correlation_heatmap(
    matrix,
    title,
    output_path,
    colorbar_label
):

    if matrix.empty:

        return


    n = len(
        matrix.columns
    )


    # Dynamic figure size
    figure_size = max(
        16,
        min(
            32,
            0.45 * n + 12
        )
    )


    plt.figure(
        figsize=(
            figure_size,
            figure_size
        )
    )


    # Avoid unreadable annotation when many descriptors
    annotate = (
        n <= 25
    )


    sns.heatmap(

        matrix,

        annot=annotate,

        fmt=".2f",

        center=0,

        cmap="coolwarm",

        linewidths=0.3,

        cbar_kws={
            "label":
                colorbar_label
        }
    )


    plt.title(
        title,
        fontsize=16
    )


    plt.xticks(
        rotation=90,
        fontsize=7
    )


    plt.yticks(
        rotation=0,
        fontsize=7
    )


    plt.tight_layout()


    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )


    plt.show()

    plt.close()


# ============================================================
# 46. PEARSON HEATMAP
# ============================================================

pearson_png = os.path.join(

    output_folder,

    "Pearson_Correlation_Heatmap.png"
)


save_correlation_heatmap(

    pearson_matrix,

    "Pearson Correlation Matrix — R3 Descriptors and %de",

    pearson_png,

    "Pearson r"
)


# ============================================================
# 47. SPEARMAN HEATMAP
# ============================================================

spearman_png = os.path.join(

    output_folder,

    "Spearman_Correlation_Heatmap.png"
)


save_correlation_heatmap(

    spearman_matrix,

    "Spearman Correlation Matrix — R3 Descriptors and %de",

    spearman_png,

    "Spearman ρ"
)


# ============================================================
# 48. GROUPED CORRELATIONS
# ============================================================

def grouped_correlations(
    descriptors,
    group_name
):

    rows = []


    for descriptor in descriptors:

        if descriptor not in corr_data.columns:

            continue


        pair = corr_data[
            [
                descriptor,
                "%de"
            ]
        ].dropna()


        n = len(
            pair
        )


        if n < 3:

            rows.append({

                "Control":
                    group_name,

                "Descriptor":
                    descriptor,

                "N":
                    n,

                "Pearson_r":
                    np.nan,

                "Pearson_p":
                    np.nan,

                "Spearman_rho":
                    np.nan,

                "Spearman_p":
                    np.nan,

                "Abs_Pearson_r":
                    np.nan,

                "Abs_Spearman_rho":
                    np.nan
            })

            continue


        x = pair[
            descriptor
        ].to_numpy(
            dtype=float
        )


        y = pair[
            "%de"
        ].to_numpy(
            dtype=float
        )


        if (
            np.unique(x).size < 2
            or
            np.unique(y).size < 2
        ):

            pr = np.nan
            pp = np.nan
            sr = np.nan
            sp = np.nan

        else:

            try:

                pr, pp = pearsonr(
                    x,
                    y
                )

            except Exception:

                pr = np.nan
                pp = np.nan


            try:

                sr, sp = spearmanr(
                    x,
                    y
                )

            except Exception:

                sr = np.nan
                sp = np.nan


        rows.append({

            "Control":
                group_name,

            "Descriptor":
                descriptor,

            "N":
                n,

            "Pearson_r":
                pr,

            "Pearson_p":
                pp,

            "Spearman_rho":
                sr,

            "Spearman_p":
                sp,

            "Abs_Pearson_r":
                (
                    abs(pr)
                    if pd.notna(pr)
                    else np.nan
                ),

            "Abs_Spearman_rho":
                (
                    abs(sr)
                    if pd.notna(sr)
                    else np.nan
                )
        })


    return pd.DataFrame(
        rows
    )


# ============================================================
# 49. CALCULATE EACH GROUP
# ============================================================

steric_corr = grouped_correlations(
    steric_descriptors,
    "Steric"
)


sasa_corr = grouped_correlations(
    sasa_descriptors,
    "SASA"
)


electronic_corr = grouped_correlations(
    electronic_descriptors,
    "Electronic"
)


geometric_corr = grouped_correlations(
    geometric_descriptors,
    "Geometric distortion"
)


# ============================================================
# 50. COMBINE GROUPS
# ============================================================

grouped_corr = pd.concat(

    [
        steric_corr,
        sasa_corr,
        electronic_corr,
        geometric_corr
    ],

    ignore_index=True
)


if not grouped_corr.empty:

    grouped_corr = (

        grouped_corr

        .sort_values(

            [
                "Control",
                "Abs_Spearman_rho"
            ],

            ascending=[
                True,
                False
            ],

            na_position="last"
        )

        .reset_index(
            drop=True
        )
    )


print("\n")
print("=" * 110)
print("GROUPED CORRELATIONS")
print("=" * 110)


display(
    grouped_corr
)


# ============================================================
# 51. TOP DESCRIPTORS — SPEARMAN
# ============================================================

print("\n")
print("=" * 110)
print("TOP DESCRIPTORS BY ABSOLUTE SPEARMAN CORRELATION")
print("=" * 110)


if not correlation_df.empty:

    display(

        correlation_df[
            [
                "Descriptor",
                "N",
                "Pearson_r",
                "Pearson_p",
                "Spearman_rho",
                "Spearman_p",
                "Abs_Spearman_rho"
            ]
        ]
        .head(20)
    )


# ============================================================
# 52. TOP DESCRIPTORS — PEARSON
# ============================================================

print("\n")
print("=" * 110)
print("TOP DESCRIPTORS BY ABSOLUTE PEARSON CORRELATION")
print("=" * 110)


if not correlation_df.empty:

    display(

        correlation_df[
            [
                "Descriptor",
                "N",
                "Pearson_r",
                "Pearson_p",
                "Spearman_rho",
                "Spearman_p",
                "Abs_Pearson_r"
            ]
        ]
        .sort_values(
            "Abs_Pearson_r",
            ascending=False,
            na_position="last"
        )
        .head(20)
    )


# ============================================================
# 53. SPECIAL CHECK — DISTAL VOLUME
# ============================================================

print("\n")
print("=" * 110)
print("R3 DISTAL VOLUME")
print("=" * 110)


distal_display_columns = [

    col

    for col in [

        "R3",
        "R3_%Vbur",
        "R3_Distal_Volume_A3",
        "Distal_Volume_Status",
        "%de"

    ]

    if col in results_df.columns
]


display(
    results_df[
        distal_display_columns
    ]
)


# ============================================================
# 54. RAW COMPLETE EXCEL
# ============================================================

raw_excel = os.path.join(

    output_folder,

    "COMPLETE_R3_ANALYSIS_WITH_DISTAL_VOLUME.xlsx"
)


results_df.to_excel(

    raw_excel,

    index=False
)


print(
    "\n✓ Raw complete data saved:"
)


print(
    raw_excel
)


# ============================================================
# 55. ANALYSIS SUMMARY TABLE
# ============================================================

analysis_summary_df = pd.DataFrame({

    "Parameter": [

        "Number of RR structures",

        "Total result columns",

        "Unique descriptors",

        "Descriptors in correlation matrix",

        "XTB descriptors",

        "SASA probe radius (A)",

        "Vbur radius (A)",

        "Vbur radii type",

        "Vbur radii scale",

        "Vbur hydrogens included",

        "R3 center atom",

        "R3 attached atom",

        "R3 Distal Volume descriptor",

        "Pearson matrix size",

        "Spearman matrix size"
    ],


    "Value": [

        len(results_df),

        len(results_df.columns),

        len(all_descriptors),

        max(
            len(corr_data.columns) - 1,
            0
        ),

        len(xtb_columns),

        DCM_PROBE_RADIUS,

        3.5,

        "bondi",

        1.17,

        False,

        "C18",

        "C44",

        "R3_Distal_Volume_A3",

        str(
            pearson_matrix.shape
        ),

        str(
            spearman_matrix.shape
        )
    ]
})


# ============================================================
# 56. FINAL EXCEL WORKBOOK
# ============================================================

from openpyxl import load_workbook
from openpyxl.styles import (
    Font,
    PatternFill,
    Alignment
)
from openpyxl.utils import (
    get_column_letter
)


excel_file = os.path.join(

    output_folder,

    "R3_Steric_SASA_Electronic_Geometric_DistalVolume_Analysis.xlsx"
)


print("\n")
print("=" * 120)
print("CREATING FINAL EXCEL WORKBOOK")
print("=" * 120)


if os.path.exists(
    excel_file
):

    os.remove(
        excel_file
    )


# ------------------------------------------------------------
# Write workbook
# ------------------------------------------------------------

with pd.ExcelWriter(

    excel_file,

    engine="openpyxl",

    mode="w"

) as writer:


    # 1
    results_df.to_excel(

        writer,

        sheet_name="All_Descriptors",

        index=False
    )


    # 2
    correlation_df.to_excel(

        writer,

        sheet_name="All_Correlations",

        index=False
    )


    # 3
    grouped_corr.to_excel(

        writer,

        sheet_name="Grouped_Correlations",

        index=False
    )


    # 4
    pearson_matrix.to_excel(

        writer,

        sheet_name="Pearson_Matrix"
    )


    # 5
    spearman_matrix.to_excel(

        writer,

        sheet_name="Spearman_Matrix"
    )


    # 6
    steric_corr.to_excel(

        writer,

        sheet_name="Steric_Correlation",

        index=False
    )


    # 7
    sasa_corr.to_excel(

        writer,

        sheet_name="SASA_Correlation",

        index=False
    )


    # 8
    electronic_corr.to_excel(

        writer,

        sheet_name="Electronic_Correlation",

        index=False
    )


    # 9
    geometric_corr.to_excel(

        writer,

        sheet_name="Geometric_Correlation",

        index=False
    )


    # 10
    descriptor_summary_df.to_excel(

        writer,

        sheet_name="Descriptor_List",

        index=False
    )


    # 11
    descriptor_qc_df.to_excel(

        writer,

        sheet_name="Descriptor_QC",

        index=False
    )


    # 12
    de_mapping_df.to_excel(

        writer,

        sheet_name="de_Mapping_QC",

        index=False
    )


    # 13
    if "XTB_Status" in results_df.columns:

        xtb_status_df = results_df[
            [
                "R3",
                "Filename",
                "XTB_Status"
            ]
        ].copy()

    else:

        xtb_status_df = pd.DataFrame()


    xtb_status_df.to_excel(

        writer,

        sheet_name="XTB_Status",

        index=False
    )


    # 14
    xtb_missing_df.to_excel(

        writer,

        sheet_name="XTB_Missing",

        index=False
    )


    # 15
    corr_data.to_excel(

        writer,

        sheet_name="Correlation_Data",

        index=False
    )


    # 16
    analysis_summary_df.to_excel(

        writer,

        sheet_name="Analysis_Summary",

        index=False
    )


# ============================================================
# 57. FORMAT EXCEL WORKBOOK
# ============================================================

print(
    "Formatting Excel workbook..."
)


wb = load_workbook(
    excel_file
)


header_fill = PatternFill(
    fill_type="solid",
    fgColor="D9EAF7"
)


header_font = Font(
    bold=True
)


header_alignment = Alignment(
    horizontal="center",
    vertical="center"
)


for ws in wb.worksheets:

    # --------------------------------------------------------
    # Freeze top row
    # --------------------------------------------------------

    ws.freeze_panes = "A2"


    # --------------------------------------------------------
    # Header formatting
    # --------------------------------------------------------

    if ws.max_row >= 1:

        for cell in ws[1]:

            cell.font = header_font

            cell.fill = header_fill

            cell.alignment = (
                header_alignment
            )


    # --------------------------------------------------------
    # Auto-filter
    # --------------------------------------------------------

    if (
        ws.max_row >= 2
        and
        ws.max_column >= 1
    ):

        ws.auto_filter.ref = (
            ws.dimensions
        )


    # --------------------------------------------------------
    # Column widths
    # --------------------------------------------------------

    for col_idx in range(

        1,

        ws.max_column + 1

    ):

        max_length = 0


        for row_idx in range(

            1,

            min(
                ws.max_row,
                100
            ) + 1

        ):

            cell = ws.cell(

                row=row_idx,

                column=col_idx

            )


            if cell.value is not None:

                cell_length = len(
                    str(cell.value)
                )


                if cell_length > max_length:

                    max_length = (
                        cell_length
                    )


        width = min(

            max(
                max_length + 2,
                10
            ),

            40

        )


        ws.column_dimensions[
            get_column_letter(
                col_idx
            )
        ].width = width


# ------------------------------------------------------------
# Guarantee visible sheet
# ------------------------------------------------------------

visible_sheets = [

    ws

    for ws in wb.worksheets

    if ws.sheet_state == "visible"

]


if len(
    visible_sheets
) == 0:

    wb.worksheets[0].sheet_state = (
        "visible"
    )

    wb.active = 0


# ------------------------------------------------------------
# Save workbook
# ------------------------------------------------------------

wb.save(
    excel_file
)


# ============================================================
# 58. VERIFY OUTPUT FILES
# ============================================================

if not os.path.exists(
    excel_file
):

    raise FileNotFoundError(
        "Final Excel workbook was not created."
    )


if not os.path.exists(
    raw_excel
):

    raise FileNotFoundError(
        "Raw Excel file was not created."
    )


if not os.path.exists(
    pearson_png
):

    raise FileNotFoundError(
        "Pearson heatmap was not created."
    )


if not os.path.exists(
    spearman_png
):

    raise FileNotFoundError(
        "Spearman heatmap was not created."
    )


# ============================================================
# 59. FINAL SUMMARY
# ============================================================

file_size_mb = (

    os.path.getsize(
        excel_file
    )

    /
    (1024 ** 2)

)


print("\n")
print("=" * 120)
print("✓✓✓ COMPLETE R3 CONTROL ANALYSIS FINISHED ✓✓✓")
print("=" * 120)


print(
    "\nNumber of RR structures:",
    len(results_df)
)


print(
    "Total result columns:",
    len(results_df.columns)
)


print(
    "Total UNIQUE descriptors:",
    len(all_descriptors)
)


print(
    "Descriptors entering correlation matrix:",
    max(
        len(corr_data.columns) - 1,
        0
    )
)


print(
    "Number of xTB descriptors:",
    len(xtb_columns)
)


print(
    "SASA probe radius:",
    DCM_PROBE_RADIUS,
    "A"
)


print(
    "R3 center:",
    "C18"
)


print(
    "R3 attached atom:",
    "C44"
)


print(
    "R3 %Vbur:",
    "R3_%Vbur"
)


print(
    "R3 Distal Volume:",
    "R3_Distal_Volume_A3"
)


print(
    "\nPearson matrix:",
    pearson_matrix.shape
)


print(
    "Spearman matrix:",
    spearman_matrix.shape
)


print(
    "\nFinal Excel workbook:"
)


print(
    excel_file
)


print(
    f"Excel size: "
    f"{file_size_mb:.2f} MB"
)


print(
    "\nRaw complete data:"
)


print(
    raw_excel
)


print(
    "\nPearson heatmap:"
)


print(
    pearson_png
)


print(
    "\nSpearman heatmap:"
)


print(
    spearman_png
)


print(
    "\nExcel sheets:"
)


for sheet in wb.sheetnames:

    print(
        "   ✓",
        sheet
    )


print("\n")
print("=" * 120)
print("ALL CALCULATIONS COMPLETED")
print("=" * 120)

R3 CONTROL ANALYSIS
Starting...


CHECKING BASIC PYTHON PACKAGES
✓ Basic packages installed/updated.


AttributeError: module 'numpy._core._multiarray_umath' has no attribute '_blas_supports_fpe'